# DeepReflecs Split Sensitivity

Aggregates DeepReflecs' 6 split-sensitivity fold runs (`scripts/deepreflecs_split_sensitivity.py`) into per-class and macro F1 mean/std tables, and compares against the baseline MLP's own cached fold results on the identical 6 splits. Everything here reads already-cached `*_val_metrics.json`/`*_test_metrics.json`/`split_sensitivity_summary.csv` files from `results/deepreflecs/` and `results/mlp/`, no retraining, no model loading. Full writeup: `deepreflecs.md`.

## Setup

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DEEPREFLECS_DIR = PROJECT_ROOT / "results" / "deepreflecs"
MLP_DIR = PROJECT_ROOT / "results" / "mlp"
CLASSES = ["car", "large_vehicle", "two_wheeler", "pedestrian", "pedestrian_group"]


def load_metrics(path: Path) -> pd.DataFrame:
    """Loads one cached *_val_metrics.json/*_test_metrics.json (precision/recall/f1/support
    per class) into a DataFrame indexed by class, in CLASSES order."""
    return pd.read_json(path, orient="index").loc[CLASSES]

## Learning rate fix

All results below (main run and the 6 folds) are from a corrected `LEARNING_RATE = 4e-5`, matching the baseline MLP's own tuned value. The original run used an untuned `1e-3` with no decay schedule, which produced a smoothly decreasing training loss but a validation accuracy that oscillated roughly 0.72 to 0.80 epoch to epoch even in the plateau region, so the reported test number depended on which epoch happened to land inside that oscillation. Per-batch evaluation and missing normalization were both ruled out as causes (val is a single full-set pass per epoch, inputs are already standardized); the fixed, non-decaying, too-high LR was the actual cause, the paper's own recipe anneals 0.01 to 0.0001 for the same reason.

After the fix, the last 20 epochs of validation accuracy have std 0.0017 (range 0.0069), a stable plateau instead of an oscillating one. Effect on the numbers: the single test split barely moved (macro F1 0.745 to 0.749, slightly better, not worse), but the 6-fold average advantage over the baseline shrank from +0.058 to +0.044, about a quarter smaller, once both models are compared at their own tuned LR. See `deepreflecs.md` for the full writeup.

## Main run (standing split)

In [2]:
main_val = load_metrics(DEEPREFLECS_DIR / "deepreflecs_val_metrics.json")
main_test = load_metrics(DEEPREFLECS_DIR / "deepreflecs_test_metrics.json")

print("val:")
print(main_val.round(3).to_string())
print(f"val macro F1: {main_val['f1'].mean():.4f}\n")

print("test:")
print(main_test.round(3).to_string())
print(f"test macro F1: {main_test['f1'].mean():.4f}")

val:
                  precision  recall     f1  support
car                   0.948   0.786  0.859    31055
large_vehicle         0.678   0.817  0.741     6247
two_wheeler           0.528   0.693  0.599     4629
pedestrian            0.585   0.889  0.706    11325
pedestrian_group      0.784   0.640  0.704    17858
val macro F1: 0.7220

test:
                  precision  recall     f1  support
car                   0.950   0.819  0.880    29958
large_vehicle         0.690   0.808  0.744     6103
two_wheeler           0.573   0.815  0.673     4879
pedestrian            0.615   0.880  0.724    11363
pedestrian_group      0.831   0.644  0.726    17420
test macro F1: 0.7494


## Split sensitivity: 6 folds, macro F1

In [3]:
fold_summary = pd.read_csv(DEEPREFLECS_DIR / "split_search" / "split_sensitivity_summary.csv").set_index("fold")
print(fold_summary.round(4).to_string())
print()
print(
    f"macro F1: mean={fold_summary['macro_f1'].mean():.4f} std={fold_summary['macro_f1'].std():.4f} "
    f"min={fold_summary['macro_f1'].min():.4f} max={fold_summary['macro_f1'].max():.4f}"
)

      max_ks  macro_f1
fold                  
0     0.1793    0.7134
1     0.6238    0.7788
2     0.2857    0.7432
3     0.3360    0.7058
4     0.2868    0.7484
5     0.2870    0.7208

macro F1: mean=0.7350 std=0.0272 min=0.7058 max=0.7788


## Per-class F1 across folds

In [4]:
per_fold_class_f1 = {
    fold: load_metrics(DEEPREFLECS_DIR / "split_search" / f"fold_{fold}" / "deepreflecs_val_metrics.json")["f1"]
    for fold in range(6)
}
per_fold_class_f1 = pd.DataFrame(per_fold_class_f1).T  # rows=fold, cols=class
per_fold_class_f1.index.name = "fold"
per_fold_class_f1.round(3)

,car,large_vehicle,two_wheeler,pedestrian,pedestrian_group
fold,,,,,
0,0.874,0.710,0.624,0.680,0.678
1,0.906,0.661,0.842,0.731,0.755
2,0.872,0.774,0.631,0.717,0.721
3,0.863,0.718,0.577,0.666,0.705
4,0.863,0.749,0.639,0.738,0.753
5,0.883,0.798,0.589,0.685,0.649


In [5]:
class_summary = pd.DataFrame({
    "mean": per_fold_class_f1.mean(),
    "std": per_fold_class_f1.std(),
    "min": per_fold_class_f1.min(),
    "max": per_fold_class_f1.max(),
})
class_summary.round(3)

,mean,std,min,max
car,0.877,0.016,0.863,0.906
large_vehicle,0.735,0.049,0.661,0.798
two_wheeler,0.650,0.097,0.577,0.842
pedestrian,0.703,0.030,0.666,0.738
pedestrian_group,0.710,0.042,0.649,0.755


## Comparison against the baseline MLP (same 6 splits)

In [6]:
baseline_fold_class_f1 = {
    fold: load_metrics(MLP_DIR / "split_search" / f"fold_{fold}" / "mlp_val_metrics.json")["f1"]
    for fold in range(6)
}
baseline_class_f1 = pd.DataFrame(baseline_fold_class_f1).T
baseline_class_f1.index.name = "fold"

paired = pd.DataFrame({
    "baseline_macro_f1": baseline_class_f1.mean(axis=1),
    "deepreflecs_macro_f1": fold_summary["macro_f1"],
})
paired["delta"] = paired["deepreflecs_macro_f1"] - paired["baseline_macro_f1"]
print(paired.round(4).to_string())
print()
print(
    f"delta: mean={paired['delta'].mean():.4f} std={paired['delta'].std():.4f} "
    f"min={paired['delta'].min():.4f} max={paired['delta'].max():.4f}"
)
print(f"DeepReflecs wins {(paired['delta'] > 0).sum()}/6 folds")

      baseline_macro_f1  deepreflecs_macro_f1   delta
fold                                                 
0                0.6788                0.7134  0.0346
1                0.7342                0.7788  0.0446
2                0.7009                0.7432  0.0422
3                0.6513                0.7058  0.0545
4                0.6953                0.7484  0.0531
5                0.6884                0.7208  0.0323

delta: mean=0.0435 std=0.0092 min=0.0323 max=0.0545
DeepReflecs wins 6/6 folds


In [7]:
class_comparison = pd.DataFrame({
    "baseline_mean": baseline_class_f1.mean(),
    "baseline_std": baseline_class_f1.std(),
    "deepreflecs_mean": per_fold_class_f1.mean(),
    "deepreflecs_std": per_fold_class_f1.std(),
})
class_comparison["delta_mean"] = class_comparison["deepreflecs_mean"] - class_comparison["baseline_mean"]
class_comparison.round(3)

,baseline_mean,baseline_std,deepreflecs_mean,deepreflecs_std,delta_mean
car,0.863,0.020,0.877,0.016,0.014
large_vehicle,0.719,0.065,0.735,0.049,0.016
two_wheeler,0.571,0.133,0.650,0.097,0.079
pedestrian,0.674,0.030,0.703,0.030,0.029
pedestrian_group,0.630,0.045,0.710,0.042,0.081


In [8]:
wins = (per_fold_class_f1 > baseline_class_f1).sum()
wins.name = "deepreflecs_wins_of_6"
print(wins.to_string())

car                 6
large_vehicle       4
two_wheeler         6
pedestrian          6
pedestrian_group    6


DeepReflecs wins every one of the 6 folds on macro F1 (mean delta +0.044, std 0.009), and its own fold-to-fold spread is essentially identical in width to the baseline's (std 0.027 both), so the gain isn't bought with less stability, but the two ranges overlap, it's a paired per-fold win, not a case of one model's range sitting entirely above the other's. Per class, `car`/`two_wheeler`/`pedestrian`/`pedestrian_group` all win 6/6 folds; `large_vehicle` only wins 4/6, its mean delta (+0.016) is noise-level, there's no real advantage or disadvantage on that class. See `deepreflecs.md` for the full writeup and mechanism discussion.